# 内存层级与数据搬运

本节把第 1 章的存储地图落到编程接口上：Tile 应该放在哪个 MemorySpace、地址如何规划对齐、数据如何搬进搬出。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. MemorySpace：Tile 的"落户"选择

`TileType` 的 `target_memory` 决定 Tile 绑定的物理缓冲区与可用数据路径：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">MemorySpace</th><th style="text-align:left;">物理缓冲区</th><th style="text-align:left;">典型用途</th></tr>
<tr><td><b>Vec</b></td><td>UB</td><td>矢量数据的输入、输出与中间结果（本课程主力）</td></tr>
<tr><td><b>Mat</b></td><td>L1 Buffer</td><td>GM 与 L0A/L0B 之间的矩阵暂存</td></tr>
<tr><td><b>Left / Right</b></td><td>L0A / L0B Buffer</td><td>矩阵乘左/右操作数</td></tr>
<tr><td><b>Acc</b></td><td>L0C Buffer</td><td>矩阵乘累加结果</td></tr>
<tr><td><b>Bias</b></td><td>BiasTable Buffer</td><td>矩阵乘融合偏置</td></tr>
<tr><td><b>Scaling</b></td><td>Fixpipe Buffer</td><td>FIX 通路的量化/反量化参数</td></tr>
<tr><td><b>ScaleLeft / ScaleRight</b></td><td>L0A_MX / L0B_MX Buffer</td><td>MX 矩阵乘的左/右量化系数矩阵</td></tr>
</table>

第 2~3 章只用到 `Vec`；矩阵相关空间（Mat/Left/Right/Acc）在第 4 章矩阵算子中登场。


## 2. TileType：完整规格

```python
tile_type = pl.TileType(
    shape=[64, 128],              # 物理 shape（当前仅支持二维正整数）
    dtype=pl.DT_FP16,
    target_memory=pl.MemorySpace.Vec,
    valid_shape=None,             # 有效区域（尾块场景，默认动态模式）
    layout=None,                  # 物理排布：ND/DN/NZ/ZN/NN/ZZ
    fractal=None,                 # 分形大小（部分空间可推导）
)
```

TileType **只描述规格不分配空间**——空间分配由下一节的 make_tile / make_tile_group 完成。TileType 还有 `pad`（无效区填充模式）与 `compact`（有效数据紧凑排布）等参数，服务于尾块与特殊排布场景；本课程用不到，需要时查 TileType 的 API 文档。


## 3. make_tile：绑定固定地址

```python
tile = pl.make_tile(tile_type, addr=0x0000)
```

三条规则：

1. `addr` 是目标 Buffer 内的**字节偏移**，必须在**编译期确定**；
2. 占用字节数由 shape × dtype 自动推导（如 64×128×FP16 = 0x4000 字节），不接受 size 参数；
3. 多个 Tile 的地址范围**不能非预期重叠**——有意复用同一地址时，开发者必须保证访问时序正确。

### 地址对齐要求

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">Buffer</th><th style="text-align:left;">addr 对齐</th></tr>
<tr><td>UB、L1 Buffer</td><td>32 字节</td></tr>
<tr><td>L0A、L0B Buffer</td><td>512 字节</td></tr>
<tr><td>L0C Buffer</td><td>64 字节</td></tr>
</table>

> 排布成 NZ/ZN 等分形格式时，不能只按逻辑元素数估算地址范围——这是第 4 章矩阵算子要小心的坑。

**make_tile 还是 make_tile_group？** 需要轮转或交给 auto_mutex 自动同步时，用 `make_tile_group`——单缓冲也只配一个 mutex ID（`mutex_ids=[0]`）；`make_tile` 创建的 Tile 不携带 mutex 元数据，跨 Pipe 依赖要开发者自己同步。


## 4. 搬运接口：load / store 家族

```python
# 按元素坐标定位（灵活，适合任意起点）
pl.load(tile, tensor, [row_offset, col_offset])    # GM -> Tile
pl.store(tensor, tile, [row_offset, col_offset])   # Tile -> GM

# 按 Tile 网格坐标定位（适合分块遍历整块 Tensor）
# pl.load_tile(...) / pl.store_tile(...) 的网格语义见 API 文档
```

约束要点：搬运范围、stride、layout 和尾块受具体接口约束；编译期可确定的非法参数由框架检查，**动态访问范围由调用方保证**。


## 5. 数据路径回放（结合第 1 章）

以 AIV Tile 计算为例的完整路径：`GM →(MTE2)→ UB(Tile) →(Vector 单元)→ UB(Tile) →(MTE3)→ GM`。本章你手工组织这条路径；第 5 章融合算子会把它与 Cube 路径（GM → L1 → L0 → Cube → L0C → GM）拼在同一个 Kernel 里。


## 6. 搬运接口的两种定位方式（详解）

`load/store`（元素坐标）与 `load_tile/store_tile`（网格坐标）的选择不是口味问题：

**元素坐标**——起点任意，适合"取 Tensor 的一个角"：

```python
# 从 x 的 (16, 32) 位置搬一块 [64, 128] 进 Tile
pl.load(tile, x, [16, 32])
# 覆盖范围：行 16~79、列 32~159（起点 + Tile shape）
```

**网格坐标**——按 Tile 尺寸分块遍历整个 Tensor（第 4 章矩阵的标准写法）：

```python
# x 按 [TILE_M, TILE_N] 划成网格，取第 (i, j) 块
pl.load_tile(tile, x, [i, j])
# 实际元素起点 = (i * TILE_M, j * TILE_N)——坐标乘法由接口代劳
```

**选型口诀**：规则分块遍历用网格坐标（少一层心算，不易错）；任意窗口用元素坐标（灵活）。两类接口处理尾块时，都需要保证有效搬运范围不越界。

**搬运量的心算模板**（每个算子设计时过一遍）：

```text
满块单次搬运字节 = Tile 行数 × Tile 列数 × dtype 字节数
Tile 块数        = ceil(M / tile_m) × ceil(N / tile_n)
总搬运字节       = 各块实际搬运字节之和（输入读取与输出写回分别统计）
```

例如 `[9,256]` FP32 按 `[4,256]` 分块，有效行数为 4、4、1，读取有效数据共 9216 B，不能按三个满块算成 12288 B。只有无尾块且每块搬运范围相同时，才可用“单块字节 × 块数”；分形排布还需按对应接口的对齐规则计算。


## 7. 地址规划完整走读（含对齐验证）

任务：为 FP32 [8, 256] 的三 Tile 布局做一次"正式"规划，把每一步依据写清楚：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">步骤</th><th style="text-align:left;">计算</th><th style="text-align:left;">结论</th></tr>
<tr><td>① 单 Tile 字节</td><td>8 × 256 × 4B = 8192B</td><td>0x2000</td></tr>
<tr><td>② 三块连续排布</td><td>0x0000 / 0x2000 / 0x4000</td><td>间隔 = 单块字节</td></tr>
<tr><td>③ 对齐验证</td><td>0x2000 = 8192 = 32 × 256 ✅</td><td>UB 32B 对齐满足</td></tr>
<tr><td>④ 总占用</td><td>3 × 0x2000 = 0x6000 = 24KB</td><td>需 < UB 可用量（950 实测确认）</td></tr>
<tr><td>⑤ mutex 分配</td><td>[0] / [1] / [2]（三块可能同时活跃）</td><td>避免无关缓冲串行化，真实数据依赖仍需等待</td></tr>
</table>

**对齐是接口约束**：`make_tile` 文档要求 UB/L1 地址 32B 对齐，L0A/L0B 512B 对齐，L0C 64B 对齐。不满足要求可能导致编译或执行错误，不能假定硬件会自动拆分传输并保持正确性。布局规划应同时满足容量、对齐与访问时序要求。

**改 dtype 重算示例**：同一布局换 FP16 → 单 Tile = 8 × 256 × 2 = 4096B = 0x1000 → 地址 0x0000/0x1000/0x2000。**每次改 dtype/shape 都要重走这张表**——把它变成肌肉记忆。


## 8. 实操：规划一个三 Tile 的地址布局

任务：为 [64, 128] FP16 的输入 x、y 与输出 out 规划 UB 地址。

```python
import pypto_pro.language as pl

tt = pl.TileType(shape=[64, 128], dtype=pl.DT_FP16, target_memory=pl.MemorySpace.Vec)

# 单块 Tile = 64*128*2 = 16384 字节 = 0x4000（天然满足 32 字节对齐）
tile_x = pl.make_tile(tt, addr=0x0000)
tile_y = pl.make_tile(tt, addr=0x4000)
tile_out = pl.make_tile(tt, addr=0x8000)
```

【待 950 验证：地址推导逻辑——0x4000 为 32 的倍数，满足 UB 对齐要求】


### 用区间检查代替“地址看起来够远”

设输入组的 Tile 为 `[4,256]` FP32，每槽 4096 B=0x1000；双槽从 0 开始，占 `[0,0x2000)`。另一个双槽输入组可从 0x2000 开始，占 `[0x2000,0x4000)`；单槽输出从 0x4000 开始，占 `[0x4000,0x5000)`。

逐个检查：每个起点满足 UB 的 32 B 对齐；按槽深度乘容量；只比较同一 MemorySpace 内的区间；最终末地址不超过目标平台容量。增加缓冲深度、改变 dtype 或 shape，都要重新算一次。仅设置不同 mutex ID 无法消除地址重叠，反过来不同地址也不意味着可以省掉生产消费依赖。


## 9. 课后练习

1. （填空）[32, 64] FP16 的 Tile 占用 ______ 字节；若三块这样的 Tile 从 0x0000 连续排布，第三块的起始地址是 ______（十六进制）。
2. （选择）L0A Buffer 上 Tile 的起始地址对齐要求是：
   A. 32 字节<br/>B. 64 字节<br/>C. 512 字节
3. （实践）把 01.06 第 1 章向量加法的 Tile 从 FP32 [8,256] 换成 FP16 [8,512]，重新推导三个 Tile Group 的地址。

In [ ]:
!cat ./answer/02.04_answer.txt

## 10. 本节小结

- `target_memory` 决定 Tile 落户哪个 Buffer；矢量计算用 `Vec`（UB）；
- `make_tile` 绑定编译期确定的字节地址，**地址不可非预期重叠**，注意对齐（UB/L1 32B、L0A/B 512B、L0C 64B）；
- `load/store` 按元素坐标定位，`load_tile/store_tile` 按网格坐标定位；
- 占用字节数 = shape 乘积 × dtype 宽度，是地址规划的出发点。

下一节：当一块 Tile 不够用时——多缓冲与轮转。